In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
train = pd.read_csv("/content/train.csv")
train.head(5)

/tmp/ipykernel_5769/587473486.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv("/content/train.csv")


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1


In [ ]:
store = pd.read_csv('/content/store.csv')
store.head(5)

,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


In [ ]:
train['Date'] = pd.to_datetime(train['Date'])

In [ ]:
train.info()

In [ ]:
df = train.merge(store, on = 'Store', how = 'left')

In [ ]:
df = df[df['Open'] == 1].copy()

In [ ]:
df['StateHoliday'] = df['StateHoliday'].replace('0', 0)

In [ ]:
df['CompetitionDistance']  = df['CompetitionDistance'].fillna(df['CompetitionDistance'].max() * 2)

In [ ]:
df['Assortment'] = df['Assortment'].fillna(df['Assortment'].mode())

In [ ]:
df['Promo2'] = df['Promo2'].fillna(0)

In [ ]:
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Week'] = df['Date'].dt.isocalendar().week.astype(int)

In [ ]:
print(f"Shape after filtering closed stores: {df.shape}")
print(f"\nNull counts:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
print(f"\nDate range: {df['Date'].min()} to {df['Date'].max()}")
print(f"Unique stores: {df['Store'].nunique()}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

daily_sales = df.groupby('Date')['Sales'].mean().reset_index()
axes[0].plot(daily_sales['Date'], daily_sales['Sales'],
             color='steelblue', linewidth=0.8, alpha=0.9)
axes[0].set_title('Daily Average Sales Over Time', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Average Sales')
axes[0].grid(True, alpha=0.3)

df['YearMonth'] = df['Date'].dt.to_period('M')
monthly_sales = df.groupby('YearMonth')['Sales'].mean().reset_index()
monthly_sales['YearMonth'] = monthly_sales['YearMonth'].astype(str)

axes[1].bar(monthly_sales['YearMonth'], monthly_sales['Sales'],
            color='steelblue', alpha=0.75)
axes[1].set_title('Monthly Average Sales', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Average Sales')
axes[1].tick_params(axis='x', rotation=90)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('01_sales_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot 1 done")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize = (14, 5))

day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
dow_sales = df.groupby('DayOfWeek')['Sales'].mean()
axes[0].bar(day_labels, dow_sales.values, color='coral', alpha=0.85, edgecolor='white')
axes[0].set_title('Average Sales by Day of Week', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Day')
axes[0].set_ylabel('Average Sales')
axes[0].grid(True, alpha=0.3, axis='y')

dow_customers = df.groupby('DayOfWeek')['Customers'].mean()
axes[1].bar(day_labels, dow_customers.values, color='mediumseagreen',
            alpha=0.85, edgecolor='white')
axes[1].set_title('Average Customers by Day of Week', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Day')
axes[1].set_ylabel('Average Customers')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('02_sales_by_dow.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot 2 done")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize = (16, 5))

promo_avg = df.groupby('Promo')['Sales'].mean()
promo = df[df['Promo'] == 1]['Sales']
no_promo = df[df['Promo'] == 0]['Sales']

df.boxplot(column = 'Sales', by = 'Promo', ax = axes[0], boxprops = dict(color = 'steelblue'), medianprops = dict(color = 'red', linewidth = 2))
axes[0].set_title('Sales Distribution: Promo vs No Promo', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Promo (0 = No, 1 = Yes)')
axes[0].set_ylabel('Sales')
plt.sca(axes[0])
plt.title('Sales Distribution: Promo vs No Promo')

axes[1].hist(no_promo, bins=50, alpha=0.6, color='salmon',
             label='No Promo', density=True)
axes[1].hist(promo, bins=50, alpha=0.6, color='steelblue',
             label='Promo', density=True)
axes[1].set_title('Sales Density: Promo vs No Promo', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Sales')
axes[1].set_ylabel('Density')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

bars = axes[2].bar(['No Promo', 'Promo'], promo_avg.values,
                    color=['salmon', 'steelblue'], alpha=0.85, edgecolor='white')
axes[2].set_title('Average Sales: Promo vs No Promo', fontsize=11, fontweight='bold')
axes[2].set_ylabel('Average Sales')
axes[2].grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, promo_avg.values):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'{val:,.0f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('03_promo_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nPromo lifts average sales by: {((promo_avg[1]/promo_avg[0])-1)*100:.1f}%")
print(df.groupby('Promo')['Sales'].agg(['mean', 'median', 'count']))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

colors = ['steelblue', 'coral', 'mediumseagreen', 'orchid']

type_sales = df.groupby('StoreType')['Sales'].mean()
axes[0,0].bar(type_sales.index, type_sales.values, color=colors, alpha=0.85, edgecolor='white')
axes[0,0].set_title('Average Sales by Store Type', fontsize=11, fontweight='bold')
axes[0,0].set_xlabel('Store Type')
axes[0,0].set_ylabel('Average Sales')
axes[0,0].grid(True, alpha=0.3, axis='y')

type_customers = df.groupby('StoreType')['Customers'].mean()
axes[0,1].bar(type_customers.index, type_customers.values, color=colors, alpha=0.85, edgecolor='white')
axes[0,1].set_title('Average Customers by Store Type', fontsize=11, fontweight='bold')
axes[0,1].set_xlabel('Store Type')
axes[0,1].set_ylabel('Average Customers')
axes[0,1].grid(True, alpha=0.3, axis='y')

assort_sales = df.groupby('Assortment')['Sales'].mean()
assort_labels = {'a': 'Basic', 'b': 'Extra', 'c': 'Extended'}
axes[1,0].bar([assort_labels.get(x, x) for x in assort_sales.index],
              assort_sales.values,
              color=colors[:len(assort_sales)], alpha=0.85, edgecolor='white')
axes[1,0].set_title('Average Sales by Assortment Type', fontsize=11, fontweight='bold')
axes[1,0].set_xlabel('Assortment')
axes[1,0].set_ylabel('Average Sales')
axes[1,0].grid(True, alpha=0.3, axis='y')

type_promo = df.groupby(['StoreType', 'Promo'])['Sales'].mean().unstack()
type_promo.plot(kind='bar', ax=axes[1,1], color=['salmon', 'steelblue'],
                alpha=0.85, edgecolor='white')
axes[1,1].set_title('Promo Effect by Store Type', fontsize=11, fontweight='bold')
axes[1,1].set_xlabel('Store Type')
axes[1,1].set_ylabel('Average Sales')
axes[1,1].legend(['No Promo', 'Promo'])
axes[1,1].tick_params(axis='x', rotation=0)
axes[1,1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('04_store_type_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot 4 done")

In [ ]:
num_cols = ['Sales', 'Customers', 'Promo', 'SchoolHoliday',
            'DayOfWeek', 'CompetitionDistance', 'Promo2', 'Month', 'Year']

corr_df = df[num_cols].copy()
corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(11, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, vmin=-1, vmax=1,
            linewidths=0.5, ax=ax, annot_kws={'size': 9})

ax.set_title('Feature Correlation Heatmap', fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)

plt.tight_layout()
plt.savefig('05_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTop correlations with Sales:")
print(corr_matrix['Sales'].sort_values(ascending=False))